# Challenge 5: The CI gate

You have an agent that scores well. A gate keeps it that way: on every change, CI reruns the evals on the **same pinned dataset version** and fails the build if scores drop below the baseline.

You'll build the gate decision, record your fixed agent as the baseline, and run the whole thing from a script with an exit code. That is exactly what a CI job needs, and the results file it leaves behind is a repeatable record of every release check.

**One cell was blank in the workshop. Here it is filled in, so every cell runs.**

In [ ]:
import os
import sys

for path in ("..", "../lab"):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv("../.env", override=True)

import lab_eval

print("project:", os.environ["LANGSMITH_PROJECT"])
print("dataset:", lab_eval.dataset_name(), "version", lab_eval.DATASET_TAG)

## 1. Read the CI script

`ci/run_evals.py` is provided. It loads the pinned dataset version, runs the agent, scores it with the evaluators you built, asks `gate.py` whether any score regressed against `baseline.json`, writes a results file, and exits `0` (pass) or `1` (fail).

It can also simulate a bad change: `--change drop-retrieval` removes the search tool from the agent, like a refactor that accidentally drops it.

In [ ]:
print(open("../ci/run_evals.py").read())

## 2. Write the gate

The gate is the decision: given this run's scores and the baseline's, which metrics regressed? It allows a small `tolerance`, because LLM output varies and one row of 8 moves a score by 0.125. A tolerance of 0.15 forgives one flipped row and catches two.

The next cell holds this code:

```python
GATE_KEYS = ["groundedness", "tool_selection"]


def find_regressions(scores, baseline, tolerance=0.15):
    '''One message per metric that fell more than `tolerance` below the baseline.'''
    problems = []
    for key in GATE_KEYS:
        got = scores.get(key, 0.0)
        if got < baseline[key] - tolerance:
            problems.append(f"{key}: {got:.2f} is below the baseline {baseline[key]:.2f}")
    return problems
```

In [ ]:
# Solution. In the workshop this cell was blank and you wrote it.
GATE_KEYS = ["groundedness", "tool_selection"]


def find_regressions(scores, baseline, tolerance=0.15):
    """One message per metric that fell more than `tolerance` below the baseline."""
    problems = []
    for key in GATE_KEYS:
        got = scores.get(key, 0.0)
        if got < baseline[key] - tolerance:
            problems.append(f"{key}: {got:.2f} is below the baseline {baseline[key]:.2f}")
    return problems

Try it on two made-up score sets, then save it where the CI script imports it from. A 0.90 is within tolerance of a 1.00 baseline. A 0.50 is not.

In [ ]:
import inspect
from pathlib import Path

good = {"groundedness": 1.0, "tool_selection": 1.0}
print(find_regressions({"groundedness": 0.90, "tool_selection": 1.0}, good))
print(find_regressions({"groundedness": 0.50, "tool_selection": 1.0}, good))

Path("../ci/gate.py").write_text(
    f"GATE_KEYS = {GATE_KEYS!r}\n\n\n{inspect.getsource(find_regressions)}"
)
print("Saved ../ci/gate.py")

## 3. Record the baseline

The baseline is the scores of the agent you are willing to ship: your `v2-fixed` experiment. It is saved as `baseline.json`. In a real repo you would commit that file, so changing the baseline is a reviewed change.

In [ ]:
lab_eval.write_baseline("v2-fixed")

## 4. Run the gate on an unchanged agent

Same agent, same dataset version. This should pass: exit code `0`. It takes about a minute.

In [ ]:
import subprocess, sys

r = subprocess.run([sys.executable, "../ci/run_evals.py", "--change", "none"], capture_output=True, text=True)
print(r.stdout + r.stderr)
print("exit code:", r.returncode)

## 5. Run the gate on a regression

Now the simulated bad change: the search tool is gone. The gate should fail: exit code `1`, with a message naming each metric that dropped.

In [ ]:
import subprocess, sys

r = subprocess.run([sys.executable, "../ci/run_evals.py", "--change", "drop-retrieval"], capture_output=True, text=True)
print(r.stdout + r.stderr)
print("exit code:", r.returncode)

## 6. The record

Each run leaves a results file with the agent change, dataset version, scores, baseline, regressions and verdict. This is the artifact to hand to a reviewer or auditor: anyone can see what was checked, against what, and what happened.

In [ ]:
import json, pathlib

for f in sorted(pathlib.Path("../ci_results").glob("*.json")):
    print(f.name)
    print(json.dumps(json.loads(f.read_text()), indent=2))
    print()

## 7. Wire it into CI

A CI job is the same command. A sketch of a GitHub Actions workflow (not run here):

```yaml
on: pull_request
jobs:
  eval-gate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: "3.12"}
      - run: pip install langsmith langchain langchain-anthropic python-dotenv
      - run: python ci/run_evals.py --change none   # non-zero exit fails the job
        env:
          LANGSMITH_API_KEY: ${{ secrets.LANGSMITH_API_KEY }}
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
      - uses: actions/upload-artifact@v4
        if: always()
        with: {name: ci_results, path: ci_results/}   # keep the record for every run
```

Set `LANGSMITH_API_KEY` as a repository secret. Because the job calls the SDK and uses an exit code, it works the same in any CI system.